# EDA REFIT — bước 1: nạp dữ liệu

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

CODE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "core" / "paths.py").is_file())
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

from src.core.paths import DATA_RAW
from src.data.refit.load import house_csv, load

ARTIFACTS = CODE_ROOT / "data" / "artifacts"
README_PATH = ARTIFACTS / "REFIT_Readme.txt"
METADATA_PATH = ARTIFACTS / "MetaData_Tables.xlsx"
CLEAN_DIR = DATA_RAW / "refit" / "zenodo-5063428"
HOUSES = [1, 2, 3, 4, 5, 11]

## 1. README: danh sách thiết bị và ghi chú

Nguồn: `data/artifacts/REFIT_Readme.txt` (README của hồ sơ Strathclyde). README mô tả các trường dữ liệu như sau.

**Tệp công suất `CLEAN_House<N>.csv`** — mỗi dòng là một thời điểm ghi:

| Trường | Ý nghĩa theo README | Đơn vị |
| --- | --- | --- |
| `Time` | Thời điểm ghi, dạng `YYYY-MM-DD HH:mm:ss` (README gọi là `DateTime`; không nêu múi giờ) | — |
| `Unix` | Dấu thời gian Unix, UTC | giây |
| `Aggregate` | Công suất tổng của hộ, đo bằng kẹp dòng (kênh 0) | W |
| `Appliance1` … `Appliance9` | Công suất của 9 Individual Appliance Monitor (IAM), kênh 1–9; thiết bị ứng với từng kênh khác nhau giữa các hộ | W |
| `Issues` | Không có trong README. Theo mã làm sạch của tác giả (GitHub `David-Murray/REFIT`), bằng 1 khi `Aggregate − ΣAppliance1..9 < 0` | 0/1 |

Các điều kiện đo README nêu:

- Chỉ đo công suất tác dụng (active power), chu kỳ khoảng 6–8 giây; các cảm biến không đồng bộ, giá trị có thể được cập nhật bất kỳ lúc nào trong 6–8 giây trước đó.
- Dữ liệu chỉ được ghi khi tải thay đổi; khoảng trống được lấp bằng giá trị đo gần nhất, trừ khi dài hơn 2 phút thì lấp bằng 0.
- Bản làm sạch lấp NaN bằng giá trị trước đó và loại giá trị IAM trên 4000 W.
- Có những đợt mất dữ liệu (đáng chú ý là tháng 2/2014).

**Danh sách thiết bị** — mỗi hộ có một khối `House N` gồm dòng `0.Aggregate, 1.<thiết bị>, …, 9.<thiết bị>` (thiết bị được biết là có đo lúc bắt đầu nghiên cứu) và mục `!NOTES` ghi thiết bị con của các "Site", thay đổi thiết bị và lưu ý khác. README cảnh báo không bảo đảm hộ gia đình không đổi thiết bị gắn vào IAM.

Ô code dưới nạp README thành hai bảng:

- `readme_appliances`: `house` (số hộ), `channel` (0 = Aggregate, 1–9 = `Appliance1..9`), `appliance` (tên thiết bị như ghi trong README).
- `readme_notes`: `house`, `note` (một dòng ghi chú, giữ nguyên văn).


In [ ]:
readme_text = README_PATH.read_text(encoding="utf-8")
house_blocks = re.split(r"^House (\d+)\s*$", readme_text.split("APPLIANCE LIST", 1)[1], flags=re.MULTILINE)[1:]

appliance_rows, note_rows = [], []
for house, block in zip(house_blocks[0::2], house_blocks[1::2]):
    listing, _, notes = block.partition("!NOTES")
    for channel, appliance in re.findall(r"(\d)\.\s*(.*?)\s*(?=,?\s*\d\.|$)", " ".join(listing.split())):
        appliance_rows.append({"house": int(house), "channel": int(channel), "appliance": appliance.strip(" ,")})
    note_rows += [{"house": int(house), "note": line.strip()} for line in notes.splitlines() if line.strip()]

readme_appliances = pd.DataFrame(appliance_rows)
readme_notes = pd.DataFrame(note_rows)
readme_appliances
